# Parser benchmark
Compare pypdf, pdfplumber, PyMuPDF, Tesseract, Docling, MarkItDown, the RapidOCR engine and docextract `/api/parse` on the same documents.

Ground truth (optional, next to each doc): `<name>.labels.json` (field recall) and `<name>.gt.txt` (CER/WER).
For `docextract`, start the stack first (`./run-all.sh`); set `DOCEXTRACT_URL` if it isn't on :8080.

For the interactive version with charts, run `bench/bench.sh`.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, '.')
from parsers import available
from run_bench import collect, run
from report import load, scorecard, by_category, facts, build_report
import pandas as pd
pd.set_option('display.max_colwidth', 80)
print('available:', list(available()))

## Pick documents and parsers

In [ ]:
DOCS = collect(['../corpus', '../test-docs'])   # files or folders
PARSERS = available()                              # or {k: v for k, v in available().items() if k in ['pypdf', 'tesseract']}
OUT = Path('results/notebook'); OUT.mkdir(parents=True, exist_ok=True)
[d.name for d in DOCS]

In [ ]:
rows = run(DOCS, PARSERS, OUT)

## Summary

In [ ]:
df = load(OUT)
scorecard(df)

In [ ]:
by_category(df).pivot(index='category', columns='parser', values='field_recall').round(2)

## Drill down
Which expected fields each parser missed, and side-by-side output for one document.

In [ ]:
df[df.status=='ok'][['parser','document','category','field_recall','table_rows','reading_order','missing']].sort_values(['document','field_recall'])

In [ ]:
DOC = DOCS[0].name
for p in PARSERS:
    f = OUT/'outputs'/p/f'{DOC}.txt'
    if f.exists():
        print(f'===== {p} =====')
        print(f.read_text()[:1500])

In [ ]:
df[df.status=='error'][['parser','document','error']]

In [ ]:
print(build_report(OUT))  # self-contained HTML for the report